In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path
import shutil
import pandas as pd
import os
import numpy as np

In [ ]:
def create_feature_dataset(participant_id):
    """
    Processes and extracts features for a single participant.
    """
    # --- 1. Load Data ---
    try:
        eeg_df = pd.read_csv(f'../stdata/1/{participant_id}_EEG.csv')
        gsr_df = pd.read_csv(f'../stdata/1/{participant_id}_GSR.csv')
        psy_df = pd.read_csv(f'../stdata/1/{participant_id}_PSY.csv')
    except FileNotFoundError:
        print(f"Files for participant {participant_id} not found. Skipping.")
        return None

    # --- 2. Clean and Prepare Timestamps ---
    eeg_df['UnixTime'] = pd.to_numeric(eeg_df['UnixTime'], errors='coerce')
    gsr_df['UnixTime'] = pd.to_numeric(gsr_df['UnixTime'], errors='coerce')
    psy_df['routineStart'] = pd.to_numeric(psy_df['routineStart'], errors='coerce')
    psy_df['routineEnd'] = pd.to_numeric(psy_df['routineEnd'], errors='coerce')

    eeg_df.dropna(subset=['UnixTime'], inplace=True)
    gsr_df.dropna(subset=['UnixTime'], inplace=True)
    psy_df.dropna(subset=['routineStart', 'routineEnd'], inplace=True)

    # --- 3. Create Cognitive Load Labels ---
    label_mapping = {1: 0, 2: 1, 3: 2} # Low, Medium, High
    psy_df['CognitiveLoad'] = psy_df['Category'].map(label_mapping)

    # --- 4. Feature Extraction Loop ---
    all_task_features = []

    for _, task in psy_df.iterrows():
        start_time = task['routineStart']
        end_time = task['routineEnd']

        # Slice the data for the current task
        eeg_slice = eeg_df[(eeg_df['UnixTime'] >= start_time) & (eeg_df['UnixTime'] <= end_time)]
        gsr_slice = gsr_df[(gsr_df['UnixTime'] >= start_time) & (gsr_df['UnixTime'] <= end_time)]

        if eeg_slice.empty or gsr_slice.empty:
            continue

        # --- EEG Feature Engineering ---
        features = {
            'Participant': participant_id,
            'TaskKey': task['Key'],
            'CognitiveLoad': task['CognitiveLoad']
        }
        
        # Define the brainwave bands and their corresponding columns
        bands = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
        electrodes = ['TP9', 'AF7', 'AF8', 'TP10']
        
        # Calculate mean and variance for each band
        for band in bands:
            band_cols = [f'{band}_{elec}' for elec in electrodes]
            # Take the mean across electrodes first, then across time
            features[f'EEG_{band}_Mean'] = eeg_slice[band_cols].mean(axis=1).mean()
            features[f'EEG_{band}_Var'] = eeg_slice[band_cols].mean(axis=1).var()

        # Calculate workload ratios
        theta_mean = eeg_slice[[f'Theta_{e}' for e in electrodes]].mean(axis=1).mean()
        alpha_mean = eeg_slice[[f'Alpha_{e}' for e in electrodes]].mean(axis=1).mean()
        beta_mean = eeg_slice[[f'Beta_{e}' for e in electrodes]].mean(axis=1).mean()
        
        # Add a small epsilon to avoid division by zero
        epsilon = 1e-6 
        features['EEG_Theta_Alpha_Ratio'] = theta_mean / (alpha_mean + epsilon)
        features['EEG_Theta_Beta_Ratio'] = theta_mean / (beta_mean + epsilon)

        # --- GSR Feature Engineering ---
        gsr_col = 'GSR Conductance CAL'
        features['GSR_Mean'] = gsr_slice[gsr_col].mean()
        features['GSR_Var'] = gsr_slice[gsr_col].var()
        
        all_task_features.append(features)

    return pd.DataFrame(all_task_features)

# --- Main Execution ---
print("Processing participant 1...")
features_df = create_feature_dataset(1)

if features_df is not None:
    print("Successfully created feature dataset!")
    print("Shape of the dataset:", features_df.shape)
    print("\nFirst 5 rows of the final feature table:")
    print(features_df.head())
    
    # Save the final dataset to a CSV file for the next step
    features_df.to_csv('../data/participant_1_features.csv', index=False)
    print("\nSaved features to 'participant_1_features.csv'")


Processing participant 1...
Successfully created feature dataset!
Shape of the dataset: (40, 17)

First 5 rows of the final feature table:
   Participant TaskKey  CognitiveLoad  EEG_Delta_Mean  EEG_Delta_Var  \
0            1   1spl1              0        0.239537       0.021580   
1            1   1spl2              0        0.725022       0.063404   
2            1  1Item1              0        0.842528       0.102015   
3            1  1Item2              0        0.843536       0.079742   
4            1  1Item3              0        0.797471       0.029748   

   EEG_Theta_Mean  EEG_Theta_Var  EEG_Alpha_Mean  EEG_Alpha_Var  \
0        0.176614       0.005094        0.414501       0.009543   
1        0.417921       0.059792        0.597913       0.051915   
2        0.515361       0.082573        0.674078       0.027990   
3        0.521588       0.021623        0.717120       0.019851   
4        0.449036       0.024646        0.638049       0.022884   

   EEG_Beta_Mean  EEG_Bet

Feature Extraction from EEG, GSR, and PSY Data
This code processes physiological data for a participant and generates features for further analysis.
Steps:

1. **Load Data**  
   - Reads three CSV files for a participant: EEG, GSR, and PSY.  
   - Files are stored in `../stdata/1/`.

2. **Clean Timestamps**  
   - Converts timestamps to numeric values.  
   - Drops rows with missing values.

3. **Create Cognitive Load Labels**  
   - Maps the `Category` column from PSY data:  
     - 1 → Low (0)  
     - 2 → Medium (1)  
     - 3 → High (2)

4. **Feature Extraction**  
   - For each task (from PSY start and end times), slices EEG and GSR data.  
   - EEG features: mean and variance of Delta, Theta, Alpha, Beta, Gamma bands.  
   - Ratios: Theta/Alpha and Theta/Beta.  
   - GSR features: mean and variance.

5. **Save Results**  
   - Stores all features in a dataframe.  
   - Saves the output as `../data/participant_1_features.csv`.
Summary
The code prepares a structured dataset by combining EEG, GSR, and PSY data, extracting statistical features, labeling tasks with cognitive load, and saving the results for further analysis.


In [ ]:
SRC_ROOT = Path("stdata")   
DST_ROOT = Path("data")     
CATEGORIES = ["EEG", "GSR", "PSY"]

for cat in CATEGORIES:
    (DST_ROOT / cat).mkdir(parents=True, exist_ok=True)

for i in range(1, 39):  
    src_folder = SRC_ROOT / str(i)

    if not src_folder.exists():
        print(f"Skipping: source folder not found -> {src_folder}")
        continue

    for cat in CATEGORIES:
        filename = f"{i}_{cat}.csv"
        src_file = src_folder / filename
        dst_file = DST_ROOT / cat / filename

        if not src_file.exists():
            print(f"Skipping (file not found): {src_file}")
            continue


        shutil.copy2(src_file, dst_file)
        print(f"Copied: {src_file} -> {dst_file}")

print("All done.")


Copied: stdata\1\1_EEG.csv -> data\EEG\1_EEG.csv
Copied: stdata\1\1_GSR.csv -> data\GSR\1_GSR.csv
Copied: stdata\1\1_PSY.csv -> data\PSY\1_PSY.csv
Copied: stdata\2\2_EEG.csv -> data\EEG\2_EEG.csv
Copied: stdata\2\2_GSR.csv -> data\GSR\2_GSR.csv
Copied: stdata\2\2_PSY.csv -> data\PSY\2_PSY.csv
Copied: stdata\3\3_EEG.csv -> data\EEG\3_EEG.csv
Copied: stdata\3\3_GSR.csv -> data\GSR\3_GSR.csv
Copied: stdata\3\3_PSY.csv -> data\PSY\3_PSY.csv
Copied: stdata\4\4_EEG.csv -> data\EEG\4_EEG.csv
Copied: stdata\4\4_GSR.csv -> data\GSR\4_GSR.csv
Copied: stdata\4\4_PSY.csv -> data\PSY\4_PSY.csv
Copied: stdata\5\5_EEG.csv -> data\EEG\5_EEG.csv
Copied: stdata\5\5_GSR.csv -> data\GSR\5_GSR.csv
Copied: stdata\5\5_PSY.csv -> data\PSY\5_PSY.csv
Copied: stdata\6\6_EEG.csv -> data\EEG\6_EEG.csv
Copied: stdata\6\6_GSR.csv -> data\GSR\6_GSR.csv
Copied: stdata\6\6_PSY.csv -> data\PSY\6_PSY.csv
Copied: stdata\7\7_EEG.csv -> data\EEG\7_EEG.csv
Copied: stdata\7\7_GSR.csv -> data\GSR\7_GSR.csv
Copied: stdata\7\7_P

>This code organizes participant data files into separate folders.  
It creates `EEG`, `GSR`, and `PSY` directories inside `data/`.  
For each participant (1 to 38), it checks if the source folder exists.  
If the CSV files are found, they are copied from `stdata` into the corresponding category folder under `data`.  
Missing folders or files are skipped with a message.  
Finally, it prints confirmation after copying all available files.


In [ ]:
# Paths
DATA_ROOT = Path("data")
EEG_PATH = DATA_ROOT / "EEG"
GSR_PATH = DATA_ROOT / "GSR"
PSY_PATH = DATA_ROOT / "PSY"

# Columns to keep
EEG_KEEP = [
    "Delta_TP9","Delta_AF7","Delta_AF8","Delta_TP10",
    "Theta_TP9","Theta_AF7","Theta_AF8","Theta_TP10",
    "Alpha_TP9","Alpha_AF7","Alpha_AF8","Alpha_TP10",
    "Beta_TP9","Beta_AF7","Beta_AF8","Beta_TP10",
    "Gamma_TP9","Gamma_AF7","Gamma_AF8","Gamma_TP10"
]

GSR_KEEP = ["GSR Resistance CAL", "GSR Conductance CAL"]

PSY_KEEP = ["routineStart", "routineEnd", "Difficulty"]

# ---- EEG ----
for i in range(1, 39):
    file = EEG_PATH / f"{i}_EEG.csv"
    if file.exists():
        df = pd.read_csv(file)
        df = df[EEG_KEEP]  # keep only required columns
        df.to_csv(file, index=False)

# ---- GSR ----
for i in range(1, 39):
    file = GSR_PATH / f"{i}_GSR.csv"
    if file.exists():
        df = pd.read_csv(file)
        df = df[GSR_KEEP]
        df.to_csv(file, index=False)

# ---- PSY ----
for i in range(1, 39):
    file = PSY_PATH / f"{i}_PSY.csv"
    if file.exists():
        df = pd.read_csv(file)
        df = df[PSY_KEEP]
        df.to_csv(file, index=False)

print("Columns dropped, only required columns saved back.")


Columns dropped, only required columns saved back.


>This code cleans EEG, GSR, and PSY data files for 38 participants.  
It defines which columns to keep for each type of data.  
For every participant file, it loads the CSV, keeps only the selected columns, and saves it back.  
This ensures only the necessary EEG bands, GSR measures, and PSY task info remain.  
Finally, it confirms that unwanted columns have been removed.


In [9]:
# Root path
DATA_ROOT = Path("..") / "data"  

# Folders
EEG_PATH = DATA_ROOT / "EEG"
GSR_PATH = DATA_ROOT / "GSR"
PSY_PATH = DATA_ROOT / "PSY"

# Columns to keep
EEG_KEEP = [
    "Delta_TP9","Delta_AF7","Delta_AF8","Delta_TP10",
    "Theta_TP9","Theta_AF7","Theta_AF8","Theta_TP10",
    "Alpha_TP9","Alpha_AF7","Alpha_AF8","Alpha_TP10",
    "Beta_TP9","Beta_AF7","Beta_AF8","Beta_TP10",
    "Gamma_TP9","Gamma_AF7","Gamma_AF8","Gamma_TP10"
]

GSR_KEEP = ["GSR Resistance CAL", "GSR Conductance CAL"]

PSY_KEEP = ["routineStart", "routineEnd", "Difficulty"]

# ---- EEG ----
for i in range(1, 39):
    file = EEG_PATH / f"{i}_EEG.csv"
    if file.exists():
        df = pd.read_csv(file)
        df.columns = df.columns.str.strip()  # remove extra spaces
        df = df[EEG_KEEP]                    # keep only required columns
        df.to_csv(file, index=False)
        print(f"Processed {file}")

# ---- GSR ----
for i in range(1, 39):
    file = GSR_PATH / f"{i}_GSR.csv"
    if file.exists():
        df = pd.read_csv(file)
        df.columns = df.columns.str.strip()
        df = df[GSR_KEEP]
        df.to_csv(file, index=False)
        print(f"Processed {file}")

# ---- PSY ----
for i in range(1, 39):
    file = PSY_PATH / f"{i}_PSY.csv"
    if file.exists():
        df = pd.read_csv(file)
        df.columns = df.columns.str.strip()
        df = df[PSY_KEEP]
        df.to_csv(file, index=False)
        print(f"Processed {file}")

print("All EEG, GSR, PSY files saved (extra columns dropped).")

C:\Users\SANIYA\AppData\Local\Temp\ipykernel_11880\1039098871.py:26: DtypeWarning: Columns (2,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processed ..\data\EEG\1_EEG.csv


C:\Users\SANIYA\AppData\Local\Temp\ipykernel_11880\1039098871.py:26: DtypeWarning: Columns (2,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processed ..\data\EEG\2_EEG.csv


C:\Users\SANIYA\AppData\Local\Temp\ipykernel_11880\1039098871.py:26: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


...

Processed ..\data\GSR\14_GSR.csv
Processed ..\data\GSR\15_GSR.csv
Processed ..\data\GSR\16_GSR.csv
Processed ..\data\GSR\17_GSR.csv
Processed ..\data\GSR\18_GSR.csv
Processed ..\data\GSR\19_GSR.csv
Processed ..\data\GSR\20_GSR.csv
Processed ..\data\GSR\21_GSR.csv
Processed ..\data\GSR\22_GSR.csv
Processed ..\data\GSR\23_GSR.csv
Processed ..\data\GSR\24_GSR.csv


C:\Users\SANIYA\AppData\Local\Temp\ipykernel_11880\1039098871.py:36: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processed ..\data\GSR\25_GSR.csv


C:\Users\SANIYA\AppData\Local\Temp\ipykernel_11880\1039098871.py:36: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processed ..\data\GSR\26_GSR.csv
Processed ..\data\GSR\27_GSR.csv
Processed ..\data\GSR\28_GSR.csv
Processed ..\data\GSR\29_GSR.csv
Processed ..\data\GSR\30_GSR.csv
Processed ..\data\GSR\31_GSR.csv
Processed ..\data\GSR\32_GSR.csv
Processed ..\data\GSR\33_GSR.csv
Processed ..\data\GSR\34_GSR.csv
Processed ..\data\GSR\35_GSR.csv
Processed ..\data\GSR\36_GSR.csv
Processed ..\data\GSR\37_GSR.csv
Processed ..\data\GSR\38_GSR.csv
Processed ..\data\PSY\1_PSY.csv
Processed ..\data\PSY\2_PSY.csv
Processed ..\data\PSY\3_PSY.csv
Processed ..\data\PSY\4_PSY.csv
Processed ..\data\PSY\5_PSY.csv
Processed ..\data\PSY\6_PSY.csv
Processed ..\data\PSY\7_PSY.csv
Processed ..\data\PSY\8_PSY.csv
Processed ..\data\PSY\9_PSY.csv
Processed ..\data\PSY\10_PSY.csv
Processed ..\data\PSY\11_PSY.csv
Processed ..\data\PSY\12_PSY.csv
Processed ..\data\PSY\13_PSY.csv
Processed ..\data\PSY\14_PSY.csv
Processed ..\data\PSY\15_PSY.csv
Processed ..\data\PSY\16_PSY.csv
Processed ..\data\PSY\17_PSY.csv
Processed ..\data\P

In [13]:
folders = {
    "EEG": "../data/EEG",
    "GSR": "../data/GSR",
    "PSY": "../data/PSY"
}

num_files = 38

for folder_name, folder_path in folders.items():
    if not os.path.exists(folder_path):
        print(f"[WARN] Folder not found: {folder_path}")
        continue

    print(f"\n=== Checking folder: {folder_name} ===")

    for i in range(1, num_files + 1):
        file_name = f"{i}_{folder_name}.csv"
        file_path = os.path.join(folder_path, file_name)
        
        if not os.path.exists(file_path):
            print(f"[WARN] File not found: {file_name}")
            continue
        
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            continue
        
        # Count nulls per column
        null_counts = df.isnull().sum().reset_index()
        null_counts.columns = ["Column Name", "Null Count"]
        
        print(f"\n{file_name} -")
        display(null_counts)
        html_table = null_counts.to_html(index=False)


=== Checking folder: EEG ===

1_EEG.csv -


,Column Name,Null Count
0,Delta_TP9,12
1,Delta_AF7,12
2,Delta_AF8,12
3,Delta_TP10,12
4,Theta_TP9,12
5,Theta_AF7,12
6,Theta_AF8,12
7,Theta_TP10,12
8,Alpha_TP9,12
9,Alpha_AF7,12



2_EEG.csv -


,Column Name,Null Count
0,Delta_TP9,209
1,Delta_AF7,209
2,Delta_AF8,209
3,Delta_TP10,209
4,Theta_TP9,209
5,Theta_AF7,209
6,Theta_AF8,209
7,Theta_TP10,209
8,Alpha_TP9,209
9,Alpha_AF7,209



3_EEG.csv -


...

,Column Name,Null Count
0,routineStart,0
1,routineEnd,0
2,Difficulty,2



37_PSY.csv -


,Column Name,Null Count
0,routineStart,0
1,routineEnd,0
2,Difficulty,1



38_PSY.csv -


,Column Name,Null Count
0,routineStart,0
1,routineEnd,0
2,Difficulty,2


In [23]:
eeg_path = "../data/EEG"
gsr_path = "../data/GSR"
psy_path = "../data/PSY"
num_files = 38

folders = {
    "EEG": eeg_path,
    "GSR": gsr_path,
    "PSY": psy_path
}

for f_type, path in folders.items():
    print(f"\n=== {f_type} Data Types ===")
    for i in range(1, num_files + 1):
        file = os.path.join(path, f"{i}_{f_type}.csv")
        if not os.path.exists(file):
            print(f"[WARN] File not found: {file}")
            continue
        df = pd.read_csv(file)
        print(f"\nFile: {i}_{f_type}.csv")
        print(df.dtypes)


=== EEG Data Types ===

File: 1_EEG.csv
Delta_TP9     float64
Delta_AF7     float64
Delta_AF8     float64
Delta_TP10    float64
Theta_TP9     float64
Theta_AF7     float64
Theta_AF8     float64
Theta_TP10    float64
Alpha_TP9     float64
Alpha_AF7     float64
Alpha_AF8     float64
Alpha_TP10    float64
Beta_TP9      float64
Beta_AF7      float64
Beta_AF8      float64
Beta_TP10     float64
Gamma_TP9     float64
Gamma_AF7     float64
Gamma_AF8     float64
Gamma_TP10    float64
Timestamp      object
dtype: object

File: 2_EEG.csv
Delta_TP9     float64
Delta_AF7     float64
Delta_AF8     float64
Delta_TP10    float64
Theta_TP9     float64
Theta_AF7     float64
Theta_AF8     float64
Theta_TP10    float64
Alpha_TP9     float64
Alpha_AF7     float64
Alpha_AF8     float64
Alpha_TP10    float64
Beta_TP9      float64
Beta_AF7      float64
Beta_AF8      float64
Beta_TP10     float64
Gamma_TP9     float64
Gamma_AF7     float64
Gamma_AF8     float64
Gamma_TP10    float64
Timestamp      object
dtyp

In [30]:
eeg_folder = "../data/EEG"
gsr_folder = "../data/GSR"
output_folder = "../data/merged_data"
os.makedirs(output_folder, exist_ok=True)

num_students = 38

for i in range(1, num_students + 1):
    print(f"Merging EEG + GSR for student {i}...")

    eeg_file = os.path.join(eeg_folder, f"{i}_EEG.csv")
    gsr_file = os.path.join(gsr_folder, f"{i}_GSR.csv")
    
    if not (os.path.exists(eeg_file) and os.path.exists(gsr_file)):
        print(f"[WARN] Missing EEG or GSR file for student {i}, skipping.")
        continue
    
    # Load files
    eeg_df = pd.read_csv(eeg_file)
    gsr_df = pd.read_csv(gsr_file)
    
    # Merge based on UnixTime (nearest match)
    if "UnixTime" in eeg_df.columns and "UnixTime" in gsr_df.columns:
        merged_df = pd.merge_asof(
            eeg_df.sort_values("UnixTime"),
            gsr_df.sort_values("UnixTime"),
            on="UnixTime",
            direction="nearest"
        )
    else:
        merged_df = pd.concat([eeg_df, gsr_df], axis=1)
    
    # Save merged CSV
    output_file = os.path.join(output_folder, f"student_{i}.csv")
    merged_df.to_csv(output_file, index=False)

print("All students merged and saved in 'merged_data' folder.")

Merging EEG + GSR for student 1...
Merging EEG + GSR for student 2...
Merging EEG + GSR for student 3...
Merging EEG + GSR for student 4...
Merging EEG + GSR for student 5...
Merging EEG + GSR for student 6...
Merging EEG + GSR for student 7...
Merging EEG + GSR for student 8...
Merging EEG + GSR for student 9...
Merging EEG + GSR for student 10...
Merging EEG + GSR for student 11...
Merging EEG + GSR for student 12...
Merging EEG + GSR for student 13...
Merging EEG + GSR for student 14...
Merging EEG + GSR for student 15...
Merging EEG + GSR for student 16...
Merging EEG + GSR for student 17...
Merging EEG + GSR for student 18...
Merging EEG + GSR for student 19...
Merging EEG + GSR for student 20...
Merging EEG + GSR for student 21...
Merging EEG + GSR for student 22...
Merging EEG + GSR for student 23...
Merging EEG + GSR for student 24...
Merging EEG + GSR for student 25...
Merging EEG + GSR for student 26...
Merging EEG + GSR for student 27...
Merging EEG + GSR for student 28...
M

In [ ]:
import pandas as pd
import numpy as np
import os
import logging

# Setup logging
logging.basicConfig(
    filename="per_trial_extraction.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Paths
merged_data_path = "merged_data"
psy_path = "PSY"
output_path = "data"
os.makedirs(output_path, exist_ok=True)

# Function to compute trial features
def compute_features(df, cols):
    stats = {}
    for col in cols:
        if col not in df.columns:
            logging.warning(f"Column {col} missing in trial slice.")
            continue
        stats[f"{col}_mean"] = df[col].mean()
        stats[f"{col}_std"] = df[col].std()
        stats[f"{col}_min"] = df[col].min()
        stats[f"{col}_max"] = df[col].max()
        stats[f"{col}_median"] = df[col].median()
    return stats

# Collect all trial-level data
all_trials = []

for pid in range(1, 39):  # 1 to 38 participants
    eeg_gsr_file = os.path.join(merged_data_path, f"student_{pid}.csv")
    psy_file = os.path.join(psy_path, f"{pid}_PSY.csv")

    if not os.path.exists(eeg_gsr_file):
        logging.error(f"Missing EEG+GSR file: {eeg_gsr_file}")
        continue
    if not os.path.exists(psy_file):
        logging.error(f"Missing PSY file: {psy_file}")
        continue

    # Load files
    try:
        eeg_gsr = pd.read_csv(eeg_gsr_file)
        psy = pd.read_csv(psy_file)
    except Exception as e:
        logging.error(f"Error reading files for participant {pid}: {e}")
        continue

    # Convert EEG+GSR timestamp to UNIX (if not already)
    try:
        eeg_gsr["UnixTime"] = pd.to_datetime(eeg_gsr["TimeStamp"], errors="coerce").astype("int64") // 10**9
    except Exception as e:
        logging.error(f"Error converting timestamps for participant {pid}: {e}")
        continue

    # Iterate over trials in PSY
    for _, trial in psy.iterrows():
        start, end = trial.get("routineStart"), trial.get("routineEnd")
        if pd.isna(start) or pd.isna(end):
            logging.warning(f"Missing start/end for trial in participant {pid}")
            continue

        trial_slice = eeg_gsr[(eeg_gsr["UnixTime"] >= start) & (eeg_gsr["UnixTime"] <= end)]
        if trial_slice.empty:
            logging.warning(f"No data found for participant {pid}, trial {trial.get('Key')}")
            continue

        # Select numeric EEG+GSR columns (exclude timestamp cols)
        numeric_cols = trial_slice.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c not in ["UnixTime"]]

        # Compute features
        trial_features = compute_features(trial_slice, numeric_cols)

        # Add identifiers + label
        trial_features.update({
            "Participant": pid,
            "TrialKey": trial.get("Key"),
            "Difficulty": trial.get("Difficulty", np.nan)
        })

        all_trials.append(trial_features)

# Save final per-trial dataset
final_df = pd.DataFrame(all_trials)
final_csv = os.path.join(output_path, "all_participants_features.csv")
final_df.to_csv(final_csv, index=False)

print(f"{final_csv}")


In [ ]:
# Load the CSV
file_path = '../data/all_partiipants_features.csv'
df = pd.read_csv(file_path)

# Fill numeric columns with mean
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Fill categorical/object columns with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Overwrite the existing file
df.to_csv(file_path, index=False)

# Check if any nulls remain
print(df.isnull().sum())

EEG_Delta_TP9_mean                0
EEG_Delta_TP9_std                 0
EEG_Delta_TP9_min                 0
EEG_Delta_TP9_max                 0
EEG_Delta_TP9_median              0
                                 ..
GSR_GSR Conductance CAL_min       0
GSR_GSR Conductance CAL_max       0
GSR_GSR Conductance CAL_median    0
Participant                       0
Difficulty                        0
Length: 112, dtype: int64


In [9]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/all_participants_features.csv')

# Define custom mapping
difficulty_mapping = {
    'Difficult': 2,
    'Medium': 1,     
    'Easy': 0
}

# Apply mapping
df['difficulty_label'] = df['CognitiveLoad'].map(difficulty_mapping)

# Save back to CSV
df.to_csv('../data/all_participants_features.csv', index=False)
